In [9]:
from pathlib import Path    
import pandas as pd
from adaptive_llm_router.config import ROOT_DIR

In [10]:
table_file = ROOT_DIR / "data" / "raw" / "model_capability_table_seed.csv"
capability_table = pd.read_csv(table_file)

In [12]:
print(capability_table.shape)
print(capability_table.iloc[0])

(100, 11)
model                                                           gpt-5.5
provider                                                         OpenAI
input_usd_per_1m                                                    2.5
output_usd_per_1m                                                  15.0
avg_latency_ms_seed                                                2500
output_tps_seed                                                      70
task_type                                                       writing
quality_easy                                                       0.96
quality_medium                                                     0.93
quality_hard                                                       0.89
notes                  Frontier default; use for hard/high-risk prompts
Name: 0, dtype: object


In [13]:
task_is_coding = capability_table[
    capability_table["task_type"] == "coding"
]

In [17]:
print(task_is_coding.shape)
print(task_is_coding)

(10, 11)
                     model         provider  input_usd_per_1m  \
4                  gpt-5.5           OpenAI             2.500   
14            gpt-5.4-mini           OpenAI             0.375   
24            gpt-5.4-nano           OpenAI             0.100   
34         claude-opus-4.7        Anthropic             5.000   
44       claude-sonnet-4.6        Anthropic             3.000   
54        claude-haiku-4.5        Anthropic             1.000   
64  gemini-3.1-pro-preview           Google             2.000   
74   gemini-3.1-flash-lite           Google             0.250   
84             deepseek-r1  DeepSeek/hosted             0.550   
94        llama-4-maverick      Meta/hosted             0.200   

    output_usd_per_1m  avg_latency_ms_seed  output_tps_seed task_type  \
4              15.000                 2500               70    coding   
14              2.250                 1600               85    coding   
24              0.625                  900              

In [18]:
quality_is_easy = task_is_coding[
    task_is_coding["quality_medium"] >= 0.9
]



In [19]:
print(quality_is_easy.shape)

(4, 11)


In [22]:
print(quality_is_easy["model"].iloc[0])

gpt-5.5


In [23]:
def cost_first_policy(prediction, capability_table):

    task_type = prediction["task_type"]
    difficulty_bucket = prediction["difficulty_bucket"]
    required_quality = prediction["required_quality"]

    estimated_input_tokens = prediction["estimated_input_tokens"]
    expected_output_tokens = prediction["expected_output_tokens"]

    task_candidates = capability_table[
        capability_table["task_type"] == task_type
    ]

    eligible_models = task_candidates[
        task_candidates[f"quality_{difficulty_bucket}"] >= required_quality
    ].copy()

    if eligible_models.empty:
        return None

    eligible_models["estimated_cost"] = (
        (estimated_input_tokens * eligible_models["input_usd_per_1m"] ) / 1_000_000 
        + (expected_output_tokens * eligible_models["output_usd_per_1m"]) / 1_000_000
    )

    sorted_models = eligible_models.sort_values(
        by='estimated_cost',
        ascending=True
    )

    return sorted_models["model"].iloc[0]

